# 01 - Exploracion: Como conseguimos los datos?

Esta notebook contiene el proceso de exploracion, testeo de endpoints de las APIs de Steam, explicando el procedimiento y decisiones tomadas

In [1]:
import requests
import json
import pandas as pd
import numpy as np
import os

## Por que Steam?

Elegimos Steam porque es una de las mayores plataformas de venta de videojuegos, y cuenta con datos accesibles desde 2009.

In [2]:
from dotenv import load_dotenv

# Cargar API key desde .env
load_dotenv()

KEY = os.environ.get('API_KEY')

## Que endpoints probamos?

Dentro del conjunto de APIs de Steam, nos encontramos muchisimos endpoints con informacion de todo tipo. En un inicio, comenzamos a probar algunos que parecian interesantes:
- ISteamChartsService/GetGamesByConcurrentPlayers
- ISteamChartsService/GetMostPlayedGames
- IStoreTopSellersService/GetWeeklyTopSellers



In [ ]:
### GAMES BY CONCURRENT PLAYERS

url = f'https://api.steampowered.com/ISteamChartsService/GetGamesByConcurrentPlayers/v1/?key={KEY}'
res = requests.get(url)
data = json.loads(res.text)['response']['ranks']
concurrent_p = pd.DataFrame.from_dict(data)
concurrent_p.set_index('rank', inplace=True)

### MOST PLAYED GAMES (popularity rn)

url = f'https://api.steampowered.com/ISteamChartsService/GetMostPlayedGames/v1/?key={KEY}&input_json=%7B%22data_request%22%3A%7B%22include_ratings%22%3Atrue%2C%22include_reviews%22%3Atrue%7D%7D'
res = requests.get(url)
data = json.loads(res.text)['response']['ranks']
most_p = pd.DataFrame.from_dict(data)
most_p.set_index('rank', inplace=True)

### TOP SELLER GAMES

url = f'https://api.steampowered.com/IStoreTopSellersService/GetWeeklyTopSellers/v1/?key={KEY}&input_json={"context":{"language":"english","country_code":"US"},"start_date":1577836800,"page_start":0,"page_count":50,"data_request":{"include_basic_info":true}}'
res = requests.get(url)
data = json.loads(res.text)['response']['ranks']
top_seller = pd.DataFrame.from_dict(data)
top_seller.set_index('rank', inplace=True)

In [4]:
all_games_id = pd.concat([concurrent_p['appid'], most_p['appid'], top_seller['appid']], axis=0).drop_duplicates().to_list()
all_games = pd.concat([concurrent_p, most_p, top_seller], axis=0)

## Decision: topsellers semanales desde 2009

Para seleccionar los videojuegos, nos basamos en una condicion simple, que podemos mantener a lo largo de la historia de Steam. Los juegos que estuvieron dentro del top sellers en cada semana desde Diciembre del 2009

El endpoint respectivo es `IStoreTopSellersService/GetWeeklyTopSellers`

Tenemos ~860 semanas, donde cada semana tenemos juegos nuevos, o repetidos (si son populares dentro de la comunidad)

El script utilizado itera las semanas desde Diciembre de 2009, a su vez revisando las varias paginas para asegurarnos de recuperar todos los datos posibles. El mismo se encuentra en `scripts/fetch_topsellers.py`

Luego creamos el script llamado `scripts/enrich_getitems.py`, con el cual enriquecemos los datos de los videojuegos seleccionados utilizando el endpoint `IStoreBrowseService/GetItems`, el cual nos da acceso a mas datos de interes, los cuales usaremos crudos, o para calcular otras features

Guardamos el resultado en un selected.csv

Recordemos que no es el dataset final, ya que aun nos faltan las variables respuesta (las reviews)

In [5]:
df = pd.read_csv("../data/selected.csv")
df.head()

,appid,name,msrp,release_ts,n_weeks,is_free
0,550,Left 4 Dead 2,9.99,1.258434e+09,400,False
1,10180,Call of Duty®: Modern Warfare® 2 (2009),19.99,1.258002e+09,225,False
2,8980,Borderlands Game of the Year,29.99,1.693487e+09,167,False
3,17460,Mass Effect (2007),29.99,1.229674e+09,117,False
4,12210,Grand Theft Auto IV: The Complete Edition,19.99,1.585073e+09,235,False


## Ultimo paso: las reviews

Finalmente, creamos el script `fetch_reviews.py`, el cual se encarga de obtener las reviews desde el siguiente endpoint `https://store.steampowered.com/appreviews/<appid>`, el cual no requiere utilizar una API KEY. Formateamos la data segun vemos conveniente, y guardamos el resultado en `data/dataset.csv`, listo para proceder con el EDA (Analisis Exploratorio de Datos)